In [2]:
# Imports and setup
import fsspec
import pyarrow.parquet as pq
import pyarrow as pa
import pyarrow.compute as pc

import os
import time
from datetime import datetime

print("All libraries imported successfully!")

All libraries imported successfully!


## Remote Schema expolaration

In [2]:
def get_column_metadata(URL, column_names=None, detailed=False):
    
    with fsspec.open(URL, "rb") as f:
        parquet_file = pq.ParquetFile(f)
        metadata = parquet_file.metadata
        schema = parquet_file.schema_arrow
        
        # Handle column selection
        if column_names is None:
            column_names = schema.names
        elif isinstance(column_names, str):
            column_names = [column_names]
        
        # Validate columns exist
        valid_columns = []
        for col_name in column_names:
            if col_name in schema.names:
                valid_columns.append(col_name)
            else:
                print(f"Warning: Column '{col_name}' not found in schema. Available columns: {schema.names}")
        
        if not valid_columns:
            print("No valid columns to analyze")
            return
        
        mode = "DETAILED" if detailed else "OVERVIEW"
        print(f"=== {mode} FOR {len(valid_columns)} COLUMN(S) ===")
        
        # Process each column
        for column_name in valid_columns:
            col_idx = schema.names.index(column_name)
            col_type = schema.field(column_name).type
            
            if detailed:
                print(f"\n{'='*50}")
                print(f"DETAILED METADATA FOR '{column_name}'")
                print(f"{'='*50}")
            else:
                print(f"\n{column_name}: {col_type}")
            
            # Initialize aggregators
            total_nulls = 0
            total_values = 0
            has_min_max = False
            global_min = None
            global_max = None
            has_distinct_count = False
            distinct_count = None
            
            # Aggregate across all row groups
            for rg_idx in range(metadata.num_row_groups):
                rg_metadata = metadata.row_group(rg_idx)
                col_metadata = rg_metadata.column(col_idx)
                rg_num_rows = rg_metadata.num_rows
                
                if col_metadata.statistics:
                    stats = col_metadata.statistics
                    
                    # Aggregate null count
                    if stats.has_null_count:
                        total_nulls += stats.null_count
                    
                    # Aggregate min/max values
                    if stats.has_min_max:
                        has_min_max = True
                        try:
                            current_min = stats.min
                            current_max = stats.max
                            
                            # Handle different data types for comparison
                            if pa.types.is_timestamp(col_type):
                                current_min = pa.scalar(current_min).cast(col_type).as_py()
                                current_max = pa.scalar(current_max).cast(col_type).as_py()
                            
                            # Update global min/max
                            if global_min is None or current_min < global_min:
                                global_min = current_min
                            if global_max is None or current_max > global_max:
                                global_max = current_max
                                
                        except (OverflowError, Exception):
                            # If we can't compare, just take the first available values
                            if global_min is None:
                                global_min = stats.min
                                global_max = stats.max
                    
                    # Handle distinct count (take max across row groups as approximation)
                    if stats.has_distinct_count:
                        has_distinct_count = True
                        if distinct_count is None or stats.distinct_count > distinct_count:
                            distinct_count = stats.distinct_count
                
                total_values += rg_num_rows
            
            # Calculate percentages
            null_percentage = (total_nulls / total_values) * 100 if total_values > 0 else 0
            non_null_count = total_values - total_nulls
            
            # Display results based on detail level
            if detailed:
                print(f"Data type: {col_type}")
                print(f"Total rows: {total_values:,}")
                print(f"Null count: {total_nulls:,} ({null_percentage:.2f}%)")
                print(f"Non-null count: {non_null_count:,}")
                
                if has_min_max:
                    print(f"\nValue Range:")
                    try:
                        if pa.types.is_timestamp(col_type):
                            print(f"  Min: {global_min}")
                            print(f"  Max: {global_max}")
                        elif pa.types.is_string(col_type) or pa.types.is_large_string(col_type):
                            min_str = str(global_min)
                            max_str = str(global_max)
                            if len(min_str) > 50:
                                min_str = min_str[:47] + "..."
                            if len(max_str) > 50:
                                max_str = max_str[:47] + "..."
                            print(f"  Min: {min_str}")
                            print(f"  Max: {max_str}")
                        else:
                            print(f"  Min: {global_min}")
                            print(f"  Max: {global_max}")
                    except Exception as e:
                        print(f"  Min: <error displaying: {e}>")
                        print(f"  Max: <error displaying: {e}>")
                else:
                    print(f"\nValue Range: No min/max statistics available")
                
                if has_distinct_count:
                    print(f"Distinct values: {distinct_count:,}")
                else:
                    print(f"Distinct values: No distinct count statistics available")
                    
            else:
                # Quick overview mode
                print(f"  Total rows: {total_values:,}")
                print(f"  Null count: {total_nulls:,} ({null_percentage:.1f}%)")
                print(f"  Non-null: {non_null_count:,}")
                
                if has_min_max:
                    try:
                        if pa.types.is_timestamp(col_type):
                            print(f"  Value range: {global_min} to {global_max}")
                        elif pa.types.is_string(col_type) or pa.types.is_large_string(col_type):
                            min_str = str(global_min)[:20] + "..." if len(str(global_min)) > 20 else str(global_min)
                            max_str = str(global_max)[:20] + "..." if len(str(global_max)) > 20 else str(global_max)
                            print(f"  Value range: '{min_str}' to '{max_str}'")
                        else:
                            print(f"  Value range: {global_min} to {global_max}")
                    except Exception:
                        print(f"  Value range: Available (display error)")
                else:
                    print(f"  Value range: No statistics available")

In [3]:
POSTS_URL = "https://bsky-data.leobalduf.com/posts.parquet"

# First, just understand the structure
with fsspec.open(POSTS_URL, "rb") as f:
    parquet_file = pq.ParquetFile(f)
    print("Posts Database")
    print("Schema:", parquet_file.schema)
    print("Number of rows:", parquet_file.metadata.num_rows)
    print("Number of row groups:", parquet_file.metadata.num_row_groups)

Posts Database
Schema: <pyarrow._parquet.ParquetSchema object at 0x7f135bacf340>
required group field_id=-1 duckdb_schema {
  optional int64 field_id=-1 did_id (Int(bitWidth=64, isSigned=true));
  optional binary field_id=-1 rkey (String);
  optional int64 field_id=-1 created_at (Timestamp(isAdjustedToUTC=true, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional binary field_id=-1 languages (JSON);
  optional binary field_id=-1 labels (JSON);
  optional binary field_id=-1 tags (JSON);
  optional binary field_id=-1 embed_type (String);
  optional group field_id=-1 embed_record {
    optional int64 field_id=-1 did_id (Int(bitWidth=64, isSigned=true));
    optional binary field_id=-1 collection (String);
    optional binary field_id=-1 rkey (String);
  }
  optional binary field_id=-1 embed_external_uri (String);
  optional binary field_id=-1 embed_images (JSON);
  optional binary field_id=-1 embed_media (JSON);
  optional binary field_id=-1 em

In [4]:
get_column_metadata(POSTS_URL, ["did_id", "created_at"], detailed=True)

=== DETAILED FOR 2 COLUMN(S) ===

DETAILED METADATA FOR 'did_id'
Data type: int64
Total rows: 1,290,686,413
Null count: 0 (0.00%)
Non-null count: 1,290,686,413

Value Range:
  Min: 3
  Max: 34251847
Distinct values: No distinct count statistics available

DETAILED METADATA FOR 'created_at'
Data type: timestamp[us, tz=UTC]
Total rows: 1,290,686,413
Null count: 76 (0.00%)
Non-null count: 1,290,686,337

Value Range:
  Min: 0001-01-01 00:00:00+00:00
  Max: 9999-12-31 23:59:59.999000+00:00
Distinct values: No distinct count statistics available


In [ ]:
PROFILES_URL = "https://bsky-data.leobalduf.com/profiles.parquet"

# First, just understand the structure
with fsspec.open(PROFILES_URL, "rb") as f:
    parquet_file = pq.ParquetFile(f)
    print("Profiles Database")
    print("Columns:", parquet_file.schema.names)
    print("Number of rows:", parquet_file.metadata.num_rows)
    print("Number of row groups:", parquet_file.metadata.num_row_groups)

Profiles Database
Columns: ['did_id', 'rkey', 'created_at', 'description', 'labels', '/', 'size', 'mimeType', '/', 'size', 'mimeType', 'joined_via_starter_pack', 'additional_fields']
Number of rows: 32170299
Number of row groups: 262


In [6]:
get_column_metadata(PROFILES_URL, ["did_id", "created_at", "joined_via_starter_pack"], detailed=True)

=== DETAILED FOR 3 COLUMN(S) ===

DETAILED METADATA FOR 'did_id'
Data type: int64
Total rows: 32,170,299
Null count: 0 (0.00%)
Non-null count: 32,170,299

Value Range:
  Min: 1
  Max: 34251851
Distinct values: No distinct count statistics available

DETAILED METADATA FOR 'created_at'
Data type: timestamp[us, tz=UTC]
Total rows: 32,170,299
Null count: 4,247,596 (13.20%)
Non-null count: 27,922,703

Value Range:
  Min: 0081-11-15 02:48:08.855000+00:00
  Max: 2954-08-31 07:36:16.393000+00:00
Distinct values: No distinct count statistics available

DETAILED METADATA FOR 'joined_via_starter_pack'
Data type: extension<arrow.json>
Total rows: 32,170,299
Null count: 1,488,624 (4.63%)
Non-null count: 30,681,675

Value Range:
  Min: application/octet-stream
  Max: text/html
Distinct values: 5


In [11]:
BLOCKS_URL = "https://bsky-data.leobalduf.com/blocks.parquet"

# First, just understand the structure
with fsspec.open(BLOCKS_URL, "rb") as f:
    parquet_file = pq.ParquetFile(f)
    print("Profiles Database")
    print("Columns:", parquet_file.schema.names)
    print("Number of rows:", parquet_file.metadata.num_rows)
    print("Number of row groups:", parquet_file.metadata.num_row_groups)

Profiles Database
Columns: ['did_id', 'rkey', 'created_at', 'subject_id']
Number of rows: 120088510
Number of row groups: 978


In [3]:
LIKES_URL = "https://bsky-data.leobalduf.com/likes.parquet"

# First, just understand the structure
with fsspec.open(LIKES_URL, "rb") as f:
    parquet_file = pq.ParquetFile(f)
    print("Profiles Database")
    print("Columns:", parquet_file.schema.names)
    print("Number of rows:", parquet_file.metadata.num_rows)
    print("Number of row groups:", parquet_file.metadata.num_row_groups)

Profiles Database
Columns: ['did_id', 'rkey', 'created_at', 'did_id', 'collection', 'rkey']
Number of rows: 6938743344
Number of row groups: 56461


# Cleaning Rules
In both posts and profiles I'm just interested in a few columns:

In [12]:
CLEANING_RULES = {
    'posts': {
        'did_id': {
            'remove_null': True,
            'remove_empty_strings': True
        },
        'created_at': {
            'remove_null': True,
            'max_date': '2025-05-14', 
            'min_date': '2020-01-01'   # Filter out very old dates
        },
    },
    'profiles': {
        'did_id': {
            'remove_null': True,
            'remove_empty_strings': True
        },
        'created_at': {
            'remove_null': True,
            'max_date': '2025-05-14', 
            'min_date': '2020-01-01'   # Filter out very old dates
        },
    },
    'blocks': {
        'did_id': {
            'remove_null': True,
            'remove_empty_strings': True
        },
        'created_at': {
            'remove_null': True,
            'max_date': '2025-05-14', 
            'min_date': '2020-01-01'   # Filter out very old dates
        },
        'subject_id': {
            'remove_null': True,
            'remove_empty_strings': True
        },
    }
}

In [4]:
def clean_table_with_rules(table, rules, table_type='default'):

    current_table = table
    applied_rules = []
    
    rule_set = rules.get(table_type, {})
    
    for column_name, column_rules in rule_set.items():
        if column_name not in current_table.column_names:
            continue
            
        temp_table = current_table
        
        if column_rules.get('remove_null'):
            mask = pc.is_valid(temp_table[column_name])
            temp_table = temp_table.filter(mask)
            applied_rules.append(f"removed_null_{column_name}")
            if temp_table.num_rows == 0:
                break  # No rows left, stop processing
        
        # Remove empty strings (May be expensive)
        if column_rules.get('remove_empty_strings') and pa.types.is_string(temp_table[column_name].type):
            mask = pc.not_equal(temp_table[column_name], "")
            temp_table = temp_table.filter(mask)
            applied_rules.append(f"removed_empty_{column_name}")
            if temp_table.num_rows == 0:
                break
        
        if column_rules.get('max_date') and pa.types.is_timestamp(temp_table[column_name].type):
            max_date = datetime.strptime(column_rules['max_date'], '%Y-%m-%d')
            max_date_scalar = pa.scalar(max_date, type=temp_table[column_name].type)
            mask = pc.less_equal(temp_table[column_name], max_date_scalar)
            temp_table = temp_table.filter(mask)
            applied_rules.append(f"max_date_{column_name}")
            if temp_table.num_rows == 0:
                break
            
        if column_rules.get('min_date') and pa.types.is_timestamp(temp_table[column_name].type):
            min_date = datetime.strptime(column_rules['min_date'], '%Y-%m-%d')
            min_date_scalar = pa.scalar(min_date, type=temp_table[column_name].type)
            mask = pc.greater_equal(temp_table[column_name], min_date_scalar)
            temp_table = temp_table.filter(mask)
            applied_rules.append(f"min_date_{column_name}")
            if temp_table.num_rows == 0:
                break
        
        current_table = temp_table
    
    return current_table, applied_rules, []

In [5]:
def clean_parquet_by_row_groups(input_path, output_path, rules, table_type='default'):
    
    print(f"Cleaning by row groups...")
    original_size = os.path.getsize(input_path)
    print(f"Input file: {original_size / (1024**3):.2f} GB")
    
    # Open Parquet file
    pf = pq.ParquetFile(input_path)
    
    # Get ALL available columns (including nested paths)
    def get_all_columns(schema):
        """Recursively get all column paths, including nested fields"""
        all_columns = []
        for field in schema:
            if pa.types.is_struct(field.type):
                # For struct fields, get all nested paths
                for nested_field in field.type:
                    nested_path = f"{field.name}.{nested_field.name}"
                    all_columns.append(nested_path)
            else:
                # Regular top-level column
                all_columns.append(field.name)
        return all_columns

    available_columns = set(get_all_columns(pf.schema_arrow))
    rule_columns = set(rules.get(table_type, {}).keys())
    columns_to_read = list(rule_columns.intersection(available_columns))

    missing_columns = rule_columns - available_columns
    if missing_columns:
        print(f"Warning: Columns not found: {missing_columns}")
    
    # Set up Parquet writer
    writer = None
    total_rows_processed = 0
    total_rows_kept = 0
    applied_rules = None
    
    # Process each row group
    for i in range(pf.num_row_groups):
        print(f"Processing row group {i+1}/{pf.num_row_groups}...", end=" ")
        
        # Read only the columns we need from this row group
        row_group_table = pf.read_row_group(i, columns=columns_to_read)
        total_rows_processed += row_group_table.num_rows
        
        # Apply cleaning rules
        cleaned_table, current_applied_rules, removal_log = clean_table_with_rules(
            row_group_table, rules, table_type
        )
        
        total_rows_kept += cleaned_table.num_rows
        
        # Store applied rules from first row group
        if applied_rules is None:
            applied_rules = current_applied_rules
            print(f"Applied rules: {applied_rules}")
        
        # Initialize writer with first cleaned table schema
        if writer is None:
            writer = pq.ParquetWriter(
                output_path,
                cleaned_table.schema,
                compression='zstd',
                use_dictionary=True,
                write_statistics=True
            )
        
        # Write cleaned row group
        if cleaned_table.num_rows > 0:
            writer.write_table(cleaned_table)
            print(f"✅ {row_group_table.num_rows} → {cleaned_table.num_rows} rows")
        else:
            print(f"❌ entire row group removed")
    
    if writer:
        writer.close()
    
    # Calculate and print summary
    final_size = os.path.getsize(output_path)
    size_reduction = original_size - final_size
    size_reduction_percent = (size_reduction / original_size * 100) if original_size > 0 else 0
    retention_rate = (total_rows_kept / total_rows_processed * 100) if total_rows_processed > 0 else 0
    
    print("\n" + "="*60)
    print("ROW GROUP CLEANING SUMMARY:")
    print("="*60)
    print(f"Row groups: {pf.num_row_groups} processed")
    print(f"Rows:       {total_rows_processed:,} → {total_rows_kept:,} ({retention_rate:.1f}% kept)")
    print(f"File size:  {original_size / (1024**3):.2f} GB → {final_size / (1024**3):.2f} GB")
    print(f"Reduction:  {size_reduction / (1024**3):.2f} GB ({size_reduction_percent:.1f}%)")
    print(f"Columns:    {len(pf.schema_arrow.names)} → {len(columns_to_read)}")
    print("="*60)
    
    return {
        'original_rows': total_rows_processed,
        'final_rows': total_rows_kept,
        'original_size': original_size,
        'final_size': final_size,
        'row_groups_processed': pf.num_row_groups
    }

# Cleaning

In [ ]:
# Profiles 
input_path = "../data/raw/profiles.parquet"
output_path = "../data/posting/cleaned/profiles.parquet"

clean_parquet_by_row_groups(
    input_path, 
    output_path,
    CLEANING_RULES,
    'profiles',
)

Cleaning by row groups...
Input file: 1.77 GB
Processing row group 1/262... Applied rules: ['removed_null_did_id', 'removed_null_created_at', 'max_date_created_at', 'min_date_created_at']
✅ 122880 → 106653 rows
Processing row group 2/262... ✅ 122880 → 106756 rows
Processing row group 3/262... ✅ 122880 → 106585 rows
Processing row group 4/262... ✅ 122880 → 106603 rows
Processing row group 5/262... ✅ 122880 → 106731 rows
Processing row group 6/262... ✅ 122880 → 106656 rows
Processing row group 7/262... ✅ 122880 → 106534 rows
Processing row group 8/262... ✅ 122880 → 106684 rows
Processing row group 9/262... ✅ 122880 → 106592 rows
Processing row group 10/262... ✅ 122880 → 106595 rows
Processing row group 11/262... ✅ 122880 → 106560 rows
Processing row group 12/262... ✅ 122880 → 106756 rows
Processing row group 13/262... ✅ 122880 → 106681 rows
Processing row group 14/262... ✅ 122880 → 106800 rows
Processing row group 15/262... ✅ 122880 → 106770 rows
Processing row group 16/262... ✅ 122880 →

{'original_rows': 32170299,
 'final_rows': 27922675,
 'original_size': 1902151733,
 'final_size': 333805410,
 'row_groups_processed': 262}

In [8]:
# Posts
input_path = "../data/raw/chunk_0_posts.parquet"
output_path = "../data/posting/cleaned/chunk_0_posts_cleaned.parquet"

clean_parquet_by_row_groups(
    input_path, 
    output_path,
    CLEANING_RULES,
    'posts',
)

Cleaning by row groups...
Input file: 0.26 GB
Processing row group 1/5... Applied rules: ['removed_null_did_id', 'removed_null_created_at', 'max_date_created_at', 'min_date_created_at']
✅ 1048576 → 1047600 rows
Processing row group 2/5... ✅ 1048576 → 1048110 rows
Processing row group 3/5... ✅ 1048576 → 1047810 rows
Processing row group 4/5... ✅ 1048576 → 1047134 rows
Processing row group 5/5... ✅ 805696 → 804009 rows

ROW GROUP CLEANING SUMMARY:
Row groups: 5 processed
Rows:       5,000,000 → 4,994,663 (99.9% kept)
File size:  0.26 GB → 0.03 GB
Reduction:  0.23 GB (86.8%)
Columns:    13 → 2


{'original_rows': 5000000,
 'final_rows': 4994663,
 'original_size': 278857072,
 'final_size': 36747531,
 'row_groups_processed': 5}

In [13]:
# Blocks
input_path = "../data/raw/blocks.parquet"
output_path = "../data/posting/cleaned/blocks.parquet"

clean_parquet_by_row_groups(
    input_path, 
    output_path,
    CLEANING_RULES,
    'blocks',
)

Cleaning by row groups...
Input file: 1.49 GB
Processing row group 1/978... Applied rules: ['removed_null_did_id', 'removed_null_created_at', 'max_date_created_at', 'min_date_created_at', 'removed_null_subject_id']
✅ 122880 → 122880 rows
Processing row group 2/978... ✅ 122880 → 122878 rows
Processing row group 3/978... ✅ 122880 → 122880 rows
Processing row group 4/978... ✅ 122880 → 122880 rows
Processing row group 5/978... ✅ 122880 → 122880 rows
Processing row group 6/978... ✅ 122880 → 122880 rows
Processing row group 7/978... ✅ 122880 → 122880 rows
Processing row group 8/978... ✅ 122880 → 122880 rows
Processing row group 9/978... ✅ 122880 → 122880 rows
Processing row group 10/978... ✅ 122880 → 122880 rows
Processing row group 11/978... ✅ 122880 → 122880 rows
Processing row group 12/978... ✅ 122880 → 122880 rows
Processing row group 13/978... ✅ 122880 → 122880 rows
Processing row group 14/978... ✅ 122880 → 122880 rows
Processing row group 15/978... ✅ 122880 → 122880 rows
Processing row

{'original_rows': 120088510,
 'final_rows': 120084926,
 'original_size': 1599106665,
 'final_size': 1552456540,
 'row_groups_processed': 978}